<a href="https://colab.research.google.com/github/pvgbabu/AWS-Sage-Maker/blob/master/lab-2-agent-loop/lab-2-agent-loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2 — The Agent Loop, Exposed

**TDWI Transform 2026 · Hands-On: Engineering Agentic AI Solutions**

**Estimated time:** 50 minutes  
**Platform:** Google Colab  
**Coding level:** Basic Python familiarity is helpful. You do not need to be an expert.

In Lab 1, you assembled a reconciliation investigator visually in Langflow.

In Lab 2, you rebuild that same basic agent in plain Python so you can see the parts that a visual framework packaged for you: the tools, state, model decision, dispatcher, stopping policy, trace, and runtime safeguards.

You are not being tested on Python syntax. Most of the implementation is already written. When you see **YOUR TURN**, the notebook will tell you exactly what to copy and exactly where to paste it.

The goal is to think like an agent engineer:

- What should the model be allowed to decide?
- What should ordinary application code decide?
- What information must survive between steps?
- How does a model decision become a real tool call?
- What does the business actually mean by "done"?
- What should happen when the agent cannot finish?
- What prevents the system from looping, repeating work, or spending indefinitely?

By the end, you will have changed the behavior of the agent **without changing the model or the evidence**. You will do it by changing the software around the model.

## Scenario Recap

Morrow Peak Outfitters is trying to explain a recurring quarterly reporting problem.

| Report | Q3 U.S. Direct Net Revenue |
|---|---:|
| Finance executive report | &#36;4,200,000 |
| Sales QBR | &#36;3,800,000 |
| **Difference** | **&#36;400,000** |

The Lab 1 investigator found one supported cause:

**Refund recognition timing explains &#36;180,000 of the &#36;400,000 difference.**

More specifically, the Finance and Sales extracts contain the same set of refunds, but they do not treat all of them the same way for Q3. Seven refunds—**RF-1007 through RF-1013**—are tied to Q3 orders but posted in Q4. **Sales includes those refunds in its Q3 reporting treatment, while Finance excludes them from Q3 because they posted in Q4.** Together, those seven differently treated refunds total **&#36;180,000**.

That finding is supported by the available data.

But it does **not** explain the entire &#36;400,000 discrepancy.

### One term we will use throughout this lab: residual

The **residual** is simply the amount of the original discrepancy that is still unexplained.

Think of it as:

**unexplained residual = total discrepancy − amount explained so far**

So in this case:

- before any cause has been accepted: &#36;400,000 − &#36;0 = **&#36;400,000 residual**
- after the refund-timing cause is accepted: &#36;400,000 − &#36;180,000 = **&#36;220,000 residual**

The residual is not another number coming from Finance or Sales. It is a business-progress measure calculated by the application.

That becomes important because an agent can find something true and still leave a large residual unexplained.

## 1. The Harness: The Software Around the Model

The language model is the reasoning component. It is **not** the entire agent system.

The **agent harness** is the software wrapped around the model that turns model judgment into controlled application behavior.

In plain language, the harness answers questions such as:

- **What can the agent reach?** Which tools and data sources are exposed?
- **What information does it carry forward?** What belongs in structured state?
- **How does a model suggestion become an action?** Who validates and executes a tool call?
- **What is the model not allowed to do?** Which capabilities are deliberately absent or blocked?
- **What counts as finished?** What business condition permits the run to stop?
- **What keeps execution bounded?** Maximum steps, repeated-call protection, timeouts, budgets, or other safeguards.
- **What can an engineer inspect afterwards?** Decisions, actions, state changes, and traces.

In Lab 1, Langflow handled many of those mechanics inside visual components. That was useful for composition, but it also meant some control decisions were less visible.

Here, the harness is intentionally small enough to read end to end:

```text
User request
     ↓
Structured investigation state
     ↓
Model proposes the next action
     ↓
Harness validates the request
     ↓
Tool registry → approved tool executes
     ↓
State is updated
     ↓
Stopping policy evaluates business progress
     ├── continue
     └── stop / unresolved / reconciled
```

This is why the controls become more apparent in Lab 2. They are no longer hidden behind a component boundary. You can point to the exact function that stores state, dispatches a tool, blocks an action, or decides whether the task is allowed to terminate.

By the end of the lab, you will change both a **business control** and an **execution safeguard** without replacing Gemini.

## 2. Set Up the Notebook

Run the next cell.

It installs the Gemini SDK and the small set of packages used in this lab. You do not need to edit the installation cell.

In [2]:
# Install only the packages this notebook needs.
# Calling pip through the active Python interpreter keeps this cell valid Python
# and makes the setup easier to reuse outside Colab.
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-U",
    "google-genai",
    "pandas",
    "pydantic",
])

print("Lab dependencies installed.")


Lab dependencies installed.


### Configure the workshop API key

Run the **next code cell exactly as written**.

After you run it, a masked input box will appear **below the cell** asking for the Gemini API key.

1. Click the masked input box.
2. Paste the temporary workshop Gemini API key provided by the instructor.
3. Press **Enter**.
4. Wait for the confirmation message.

Do **not** paste the key into the Python code itself.

The prompt behaves like a password field, so the key will not be displayed back to you or saved in the notebook source.

In [4]:
from getpass import getpass
from google import genai

# Keep credentials outside the notebook source so the file can be shared safely.
GEMINI_API_KEY = getpass("Enter the workshop Gemini API key: ")

# Use the same workshop model as Lab 1.
MODEL_NAME = "gemini-3.1-flash-lite"

# One client is used for all model calls in this notebook.
client = genai.Client(api_key=GEMINI_API_KEY)

print(f"Gemini client configured with model: {MODEL_NAME}")

Enter the workshop Gemini API key: ··········
Gemini client configured with model: gemini-3.1-flash-lite


## 3. Load the Lab Data

The notebook first tries to load the two extracts from the workshop GitHub repository.

Before the workshop, the instructor will place the final Finance and Sales CSVs in:

`lab-2-agent-loop/data/`

If the repository copy is unavailable, the cell will prompt you to upload the two CSVs manually.

In [5]:
import io
import pandas as pd
import requests

# Keep repository paths in one place so file locations are easy to change.
REPO_RAW_BASE = (
    "https://raw.githubusercontent.com/"
    "kieDotson/tdwi-agentic-ai-workshop/main/"
    "lab-2-agent-loop/data"
)

FINANCE_FILE = "morrow_peak_finance_q3_reconciliation_extract.csv"
SALES_FILE = "morrow_peak_sales_q3_reconciliation_extract.csv"


def load_csv_from_repo(filename):
    """Load one workshop CSV from the public GitHub repository."""
    url = f"{REPO_RAW_BASE}/{filename}"
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    return pd.read_csv(io.BytesIO(response.content))


try:
    finance_df = load_csv_from_repo(FINANCE_FILE)
    sales_df = load_csv_from_repo(SALES_FILE)
    print("Loaded Finance and Sales extracts from GitHub.")

except Exception:
    # Manual upload is a recovery path if GitHub is unavailable.
    from google.colab import files

    print("Repository files were not available.")
    print("Upload the Finance and Sales CSVs when prompted.")
    uploaded = files.upload()

    finance_df = pd.read_csv(io.BytesIO(uploaded[FINANCE_FILE]))
    sales_df = pd.read_csv(io.BytesIO(uploaded[SALES_FILE]))

print(f"Finance rows: {len(finance_df):,}")
print(f"Sales rows:   {len(sales_df):,}")

Loaded Finance and Sales extracts from GitHub.
Finance rows: 62
Sales rows:   62


### Quick check

We will use the `REPORT_SUMMARY` row as the authoritative reported KPI, just as we did in Lab 1. The detailed refund records give the investigator comparable evidence to inspect.

When you run the next cell, you will also see several cells displayed as `NaN`.

`NaN` stands for **Not a Number**, but in this dataset it usually means **this field is blank or does not apply to this particular record type**. It is not an error, and it should not be treated as zero.

The extract contains different kinds of rows in the same table:

- A `REPORT_SUMMARY` row stores the team’s **official Q3 reported revenue**, so fields such as refund ID, event date, and refund amount are not populated.
- `REVENUE_SAMPLE` rows contain **sample transaction activity**, so they do not repeat the official report-level Q3 revenue value.
- `REFUND` rows contain **refund IDs, dates, inclusion behavior, and amounts**, but they also do not repeat the report-level KPI.

Pandas uses `NaN` to represent those empty numeric cells when it loads the CSV.

So when you inspect the table, read the `record_type` column first. That tells you **what kind of record you are looking at and which fields should contain values**.

The `REPORT_SUMMARY` row is the **only row we use for the official reported Q3 revenue**. We do **not** add up the detailed rows to recreate that KPI.ou are looking at and which fields should contain values.

In [6]:
display(finance_df.head(5))
display(sales_df.head(5))

,record_id,record_type,extract_scope,population_complete,target_reporting_period,reported_q3_net_revenue_usd,event_date,event_period,original_order_date,original_order_period,reporting_group,refund_id,raw_amount_usd,included_in_q3_report,reporting_amount_usd,source_view
0,FIN-SUMMARY-Q3,REPORT_SUMMARY,reported_metric,True,2026-Q3,4200000.0,NaN,NaN,NaN,NaN,U.S. Direct,NaN,NaN,NaN,NaN,Finance
1,FIN-REV-001,REVENUE_SAMPLE,partial_revenue_detail,False,2026-Q3,NaN,2026-07-01,2026-Q3,2026-07-01,2026-Q3,Online DTC,NaN,29400.0,True,29400.0,Finance
2,FIN-REV-002,REVENUE_SAMPLE,partial_revenue_detail,False,2026-Q3,NaN,2026-07-02,2026-Q3,2026-07-02,2026-Q3,Retail Stores,NaN,21900.0,True,21900.0,Finance
3,FIN-REV-003,REVENUE_SAMPLE,partial_revenue_detail,False,2026-Q3,NaN,2026-07-04,2026-Q3,2026-07-04,2026-Q3,Marketplace,NaN,36900.0,True,36900.0,Finance
4,FIN-REV-004,REVENUE_SAMPLE,partial_revenue_detail,False,2026-Q3,NaN,2026-07-06,2026-Q3,2026-07-06,2026-Q3,Other Direct,NaN,36100.0,True,36100.0,Finance


,record_id,record_type,extract_scope,population_complete,target_reporting_period,reported_q3_net_revenue_usd,event_date,event_period,original_order_date,original_order_period,reporting_group,refund_id,raw_amount_usd,included_in_q3_report,reporting_amount_usd,source_view
0,SAL-SUMMARY-Q3,REPORT_SUMMARY,reported_metric,True,2026-Q3,3800000.0,NaN,NaN,NaN,NaN,U.S. Direct,NaN,NaN,NaN,NaN,Sales
1,SAL-REV-001,REVENUE_SAMPLE,partial_revenue_detail,False,2026-Q3,NaN,2026-07-01,2026-Q3,2026-07-01,2026-Q3,Mobile App,NaN,31300.0,True,31300.0,Sales
2,SAL-REV-002,REVENUE_SAMPLE,partial_revenue_detail,False,2026-Q3,NaN,2026-07-02,2026-Q3,2026-07-02,2026-Q3,Marketplace,NaN,29100.0,True,29100.0,Sales
3,SAL-REV-003,REVENUE_SAMPLE,partial_revenue_detail,False,2026-Q3,NaN,2026-07-04,2026-Q3,2026-07-04,2026-Q3,Stores,NaN,24800.0,True,24800.0,Sales
4,SAL-REV-004,REVENUE_SAMPLE,partial_revenue_detail,False,2026-Q3,NaN,2026-07-06,2026-Q3,2026-07-06,2026-Q3,Web,NaN,20900.0,True,20900.0,Sales


## 4. Tools: Plain Python Functions With Descriptions

In Langflow, you connected tools to the Agent component.

Here, a tool is visible as ordinary application code.

Each tool has two practical pieces:

1. **the function** — the code that actually retrieves or performs something; and
2. **the description** — the information the model uses to understand when that capability is appropriate.

The two functions below are read-only data tools. One retrieves Finance evidence and the other retrieves Sales evidence.

Notice that the tools return a **compact evidence package** rather than dumping every row in the source into the model's context. Good agent engineering is partly about deciding what a model actually needs to see.

In [7]:
def load_finance_q3_revenue():
    """
    Return Finance's reported Q3 revenue and comparable refund evidence.

    This tool is intentionally read-only. The agent can inspect evidence
    through the function but cannot modify the underlying source.
    """

    # Pull the authoritative KPI from the report summary.
    summary = finance_df.loc[
        finance_df["record_type"] == "REPORT_SUMMARY",
        ["reported_q3_net_revenue_usd"],
    ].iloc[0].to_dict()

    # Return only the refund fields needed for this investigation.
    # Smaller tool outputs reduce irrelevant context and make traces easier to inspect.
    refunds = finance_df.loc[
        finance_df["record_type"] == "REFUND",
        [
            "refund_id",
            "event_date",
            "event_period",
            "original_order_period",
            "included_in_q3_report",
            "reporting_amount_usd",
        ],
    ].to_dict(orient="records")

    return {
        "source": "Finance",
        "reported_q3_net_revenue_usd": int(
            summary["reported_q3_net_revenue_usd"]
        ),
        "refunds": refunds,
    }


def load_sales_q3_revenue():
    """
    Return Sales' reported Q3 revenue and comparable refund evidence.

    The output shape matches the Finance tool so the harness can process both
    sources consistently even though the business views differ.
    """

    summary = sales_df.loc[
        sales_df["record_type"] == "REPORT_SUMMARY",
        ["reported_q3_net_revenue_usd"],
    ].iloc[0].to_dict()

    refunds = sales_df.loc[
        sales_df["record_type"] == "REFUND",
        [
            "refund_id",
            "event_date",
            "event_period",
            "original_order_period",
            "included_in_q3_report",
            "reporting_amount_usd",
        ],
    ].to_dict(orient="records")

    return {
        "source": "Sales",
        "reported_q3_net_revenue_usd": int(
            summary["reported_q3_net_revenue_usd"]
        ),
        "refunds": refunds,
    }

### The tool registry

The **tool registry** is both:

- an inventory of the tools available to this agent; and
- a capability boundary that maps a model-requested tool name to the application function that is allowed to run.

For this lab, the registry contains two tools:

- `load_finance_q3_revenue`
- `load_sales_q3_revenue`

The model does **not** receive arbitrary access to Python, your notebook, or every possible data source. It can request only the capabilities we intentionally register.

That matters operationally. Every tool you expose expands what the agent can reach and, in a production system, potentially expands its permissions and blast radius.

Also notice the descriptions. The model chooses tools from their names and descriptions; it does not inspect the underlying Python implementation before deciding what to call.

In [8]:
# The registry is the capability boundary for this teaching agent.
# The model may request these actions, but it cannot execute arbitrary Python.
TOOLS = {
    "load_finance_q3_revenue": {
        "description": (
            "Loads the Finance Q3 revenue extract. Use this tool when you need "
            "Finance-reported Q3 revenue, Finance refund treatment, or evidence "
            "from the Finance reporting source."
        ),
        "function": load_finance_q3_revenue,
    },
    "load_sales_q3_revenue": {
        "description": (
            "Loads the Sales Q3 revenue extract. Use this tool when you need "
            "Sales-reported Q3 revenue, Sales refund treatment, or evidence "
            "from the Sales reporting source."
        ),
        "function": load_sales_q3_revenue,
    },
}

for name, tool in TOOLS.items():
    print(f"{name}: {tool['description']}")

load_finance_q3_revenue: Loads the Finance Q3 revenue extract. Use this tool when you need Finance-reported Q3 revenue, Finance refund treatment, or evidence from the Finance reporting source.
load_sales_q3_revenue: Loads the Sales Q3 revenue extract. Use this tool when you need Sales-reported Q3 revenue, Sales refund treatment, or evidence from the Sales reporting source.


## Engineering Checkpoint 1 — Predict Before You Run

Before we build the loop, inspect the tool functions and answer these questions with the person next to you or in your notes:

1. Which values come directly from source data?
2. Which values will need to be calculated by the application?
3. Which decisions actually require model judgment?
4. What information should survive from one model step to the next?

Keep your answers in mind. You will compare them with the state object next.

## 5. State: What the System Knows About the Task

**State** is the structured information the harness keeps as the investigation moves from one step to the next.

For this reconciliation, state tracks:

- the Finance and Sales reported figures;
- the total discrepancy;
- how much has been explained;
- the **unexplained residual**;
- which sources have already been used;
- findings collected so far; and
- how many execution steps have occurred.

### Why state is important

A multi-step agent needs a reliable way to remember what has already happened.

Without explicit state, important facts can live only inside conversation text or model context. That makes it much easier for a system to:

- forget that a source was already queried;
- repeat the same action;
- lose track of an amount discovered several steps earlier;
- perform inconsistent arithmetic from one turn to the next;
- declare completion without comparing progress to the original objective; or
- become difficult for an engineer to inspect and test.

Structured state gives the **application** a durable, machine-checkable record of business progress.

### What the residual means inside state

At first, once both report totals are known, the total gap is &#36;400,000 and the explained amount is still &#36;0. Therefore the unexplained residual is also &#36;400,000.

That does **not** mean the agent failed to notice the refund evidence. It means the harness has not yet accepted a supported finding and applied its amount to the business ledger.

Once the &#36;180,000 refund-timing finding is accepted:

**&#36;400,000 total gap − &#36;180,000 explained = &#36;220,000 unexplained residual**

We keep that arithmetic in Python instead of asking the model to repeatedly recalculate it from prose.

In [9]:
def new_state():
    """Create a clean state object for one reconciliation run."""

    return {
        # Business facts retrieved from authoritative sources.
        "finance_reported": None,
        "sales_reported": None,

        # Deterministic business state.
        # These values are calculated by Python instead of repeatedly asking
        # the LLM to infer arithmetic from prose.
        "total_gap": None,
        "explained_amount": 0,
        "residual": None,

        # Evidence and execution history carried across agent steps.
        "findings": [],
        "sources_used": [],
        "step_count": 0,
    }


def update_gap(state):
    """
    Recalculate the discrepancy and residual from structured state.

    The model interprets evidence. Python owns the arithmetic invariant:

        residual = total discrepancy - amount explained
    """

    if (
        state["finance_reported"] is not None
        and state["sales_reported"] is not None
    ):
        state["total_gap"] = abs(
            state["finance_reported"]
            - state["sales_reported"]
        )

        state["residual"] = (
            state["total_gap"]
            - state["explained_amount"]
        )

    return state

### Compare Your Prediction

Look back at Engineering Checkpoint 1.

The design is intentionally split:

- **Source facts** come from the Finance and Sales tools.
- **Arithmetic** such as the total gap and unexplained residual belongs to deterministic Python.
- **Judgment** such as interpreting why matched refunds were treated differently belongs to the model.
- **Execution history** such as which sources have already been called belongs to state.

A strong agent system does not send every decision to the model. It puts each responsibility in the part of the system best suited to own it.

## 6. The Model's Job: Choose the Next Action

For this lab, the model's job is deliberately narrow.

On each pass, Gemini must return one structured decision:

- load the Finance source;
- load the Sales source; or
- return one evidence-supported finding.

When it returns a finding, it must provide:

- the cause;
- the amount associated with that cause; and
- a concise explanation of the evidence supporting it.

The model proposes the next move. It does **not** execute arbitrary application code, calculate the official residual, or unilaterally decide that the business objective is complete.

That separation is one of the central engineering ideas in this lab: use the model for judgment, and use the harness for enforceable control.

In [10]:
from typing import Literal, Optional
from pydantic import BaseModel, Field
from google.genai import types


class AgentDecision(BaseModel):
    """The only decision shape the model is allowed to return."""

    action: Literal[
        "load_finance_q3_revenue",
        "load_sales_q3_revenue",
        "final",
    ]

    reason: str = Field(
        description="Short explanation of why this is the next action."
    )

    cause: Optional[str] = Field(
        default=None,
        description="Evidence-supported cause when action is final.",
    )

    amount_usd: Optional[int] = Field(
        default=None,
        description="Amount associated with the supported cause when action is final.",
    )

    explanation: Optional[str] = Field(
        default=None,
        description="Concise business explanation when action is final.",
    )

In [11]:
import json


SYSTEM_INSTRUCTIONS = """
You are the Reconciliation Investigator for Morrow Peak Outfitters.

Investigate the Q3 U.S. Direct reporting discrepancy using the available
Finance and Sales reporting sources.

Rules:
- Inspect both Finance and Sales before returning a final finding.
- Do not request the same reporting source more than once.
- Use the reported summary values as the official Q3 figures.
- Base every claim on evidence returned by the tools.
- Do not invent missing information.
- Compare refund dates, periods, inclusion behavior, shared identifiers,
  and amounts when useful.
- Once you identify one specific, evidence-supported cause and the amount
  associated with it, you may return a final finding.
- When returning a final finding, make the evidence concrete. State which
  refund records are treated differently, which report includes or excludes
  them for Q3, why the period treatment differs, and how the affected amounts
  support the amount_usd value. Mention relevant refund IDs when available.
"""


def model_decision(state, evidence):
    """
    Ask Gemini for exactly one next action.

    The model receives the instructions, tool descriptions, current state,
    and evidence collected so far. It proposes an action; it does not execute
    application code itself.
    """

    available_tools = {
        name: meta["description"]
        for name, meta in TOOLS.items()
    }

    prompt = f"""
{SYSTEM_INSTRUCTIONS}

Available tools:
{json.dumps(available_tools, indent=2)}

Current investigation state:
{json.dumps(state, indent=2)}

Evidence collected so far:
{json.dumps(evidence, indent=2, default=str)}

Choose the single next action.
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            # Structured output gives the application something it can validate.
            response_mime_type="application/json",
            response_schema=AgentDecision,
            # Keep classroom runs reasonably deterministic.
            temperature=0,
        ),
    )

    return AgentDecision.model_validate_json(response.text)

## 7. Tool Dispatch: Turning a Model Decision Into an Action

The model returns a **proposal** such as:

`load_finance_q3_revenue`

The **dispatcher** is application code that decides what that proposal actually maps to.

It:

1. checks that the requested tool exists in the registry;
2. executes the approved Python function;
3. stores the returned evidence;
4. promotes authoritative values into structured state; and
5. recalculates business progress when new source data arrives.

This boundary is important.

The model can say, “I want to use the Finance tool.” It does not get to invent a new function name and have the notebook execute arbitrary code. The harness mediates the action.

In [12]:
def dispatch_tool(tool_name, state, evidence):
    """Execute one allowed tool and update the investigation state."""

    # Never trust a model-produced tool name without checking the registry.
    tool = TOOLS.get(tool_name)

    if tool is None:
        raise ValueError(f"Unknown tool requested: {tool_name}")

    # Execute the application-owned function.
    result = tool["function"]()

    # Save evidence so later model calls can reason over what was already found.
    evidence[tool_name] = result

    # Promote authoritative source values into structured business state.
    if result["source"] == "Finance":
        state["finance_reported"] = result["reported_q3_net_revenue_usd"]
    elif result["source"] == "Sales":
        state["sales_reported"] = result["reported_q3_net_revenue_usd"]

    # Record source usage for both auditability and later safeguards.
    if tool_name not in state["sources_used"]:
        state["sources_used"].append(tool_name)

    # Recalculate deterministic business state whenever new source data arrives.
    update_gap(state)

    return result

## 8. A Readable Trace

Agent systems become difficult to debug when all you see is the final answer.

For this lab, every step prints:

- the model's proposed action;
- its reason for choosing that action;
- which tool actually executed;
- the current business state; and
- the stopping-policy result.

This is a lightweight trace.

It lets you inspect the path the agent took rather than guessing from the final prose.

As you run the notebook, pay attention to how the state changes **before and after the final supported finding**. In particular, watch `Explained so far` and `Unexplained residual`.

In [13]:
def money(value):
    """Format optional numeric state without breaking when a value is unknown."""
    return "—" if value is None else f"${value:,.0f}"


def print_state(state):
    """Print the business state in a human-readable trace."""

    print("STATE")
    print(f"  Finance reported:  {money(state['finance_reported'])}")
    print(f"  Sales reported:    {money(state['sales_reported'])}")
    print(f"  Total discrepancy: {money(state['total_gap'])}")
    print(f"  Explained so far:  {money(state['explained_amount'])}")
    print(f"  Unexplained residual: {money(state['residual'])}")
    print(f"  Sources used:      {state['sources_used']}")
    print()


def sources_exhausted(state):
    """Return True when every currently available business source has been used."""
    return set(state["sources_used"]) == set(TOOLS.keys())

## 9. The Inherited Stopping Policy

The **first stopping policy in this notebook** intentionally mirrors the behavior you saw in Lab 1.

Its rule is essentially:

> If the model returns one supported cause with an amount, accept that as sufficient reason to stop.

That sounds reasonable until you separate two different ideas:

- **finding something true**
- **finishing the business task**

A stopping policy is one of the most consequential pieces of an agent harness because it turns intermediate progress into a terminal decision.

If the stopping rule is too weak, the model can produce a perfectly valid finding and the application can still stop too early.

Read the function below before running it.

Notice what it checks—and what it never checks.

In [14]:
def stop_when_model_has_supported_finding(state, decision):
    """
    Stop as soon as the model returns one supported finding.

    This is intentionally the inherited policy for the first run.
    """

    if (
        decision.action == "final"
        and decision.cause
        and decision.amount_usd is not None
    ):
        return True, "finding_returned"

    return False, "continue"

## Engineering Checkpoint 2 — Diagnose the Definition of Done

Before running the agent, answer:

1. What exact condition causes this stopping function to return `True`?
2. Does it inspect `state["residual"]`?
3. Does it distinguish between **found a supported cause** and **finished the reconciliation**?
4. If the model explains &#36;180,000 of a &#36;400,000 discrepancy, what should the business state say?
5. What business condition would you expect a stronger stopping policy to inspect?

Do not change the function yet.

The point is to learn to read stopping logic as a business rule, not merely as control-flow syntax.

## 10. Run the Agent Loop

This function is the complete control loop for the Lab 2 agent.

On each step, it does the following:

1. asks the model for one next action;
2. prints the model's decision and reason;
3. checks whether a requested tool is permitted;
4. executes the tool through the dispatcher;
5. updates evidence and structured state;
6. recalculates the total discrepancy and unexplained residual;
7. accepts a final model finding only after both required sources have been inspected;
8. applies the selected stopping policy; and
9. enforces a hard maximum-step limit even if everything else goes wrong.

That sequence is the harness in motion:

**model proposal → validation → action → state update → business stop check**

Read the comments as you scan the function. You do not need to memorize the Python. Focus on who owns each decision.

In [15]:
def run_agent(stop_policy, tool_guard=None, max_steps=8):
    """
    Run the reconciliation agent with explicit execution controls.

    stop_policy:
        Business rule that decides whether a final finding should terminate the run.

    tool_guard:
        Optional operational rule that can allow or block a requested tool call.

    max_steps:
        Hard execution limit. This protects cost and runtime even if other controls fail.
    """

    state = new_state()
    evidence = {}

    # A hard step cap is an operational safety boundary.
    # It should not be confused with a business definition of success.
    for step in range(1, max_steps + 1):
        state["step_count"] = step

        print("=" * 70)
        print(f"STEP {step}")

        # The model proposes exactly one next action.
        decision = model_decision(state, evidence)

        print(f"Decision: {decision.action}")
        print(f"Reason:   {decision.reason}")
        print()

        # PATH 1: The model requested a tool.
        if decision.action in TOOLS:

            # The harness may veto a model-proposed action before execution.
            if tool_guard is not None:
                allowed, guard_status = tool_guard(
                    decision.action,
                    state,
                )

                if not allowed:
                    print(f"TOOL CALL BLOCKED: {guard_status}")
                    print()

                    # Record the blocked action so the next model step can see
                    # that the request was rejected instead of repeating blindly.
                    evidence.setdefault("_guard_events", []).append({
                        "tool": decision.action,
                        "status": guard_status,
                    })
                    continue

            result = dispatch_tool(
                decision.action,
                state,
                evidence,
            )

            print(f"Tool executed: {decision.action}")
            print(f"Source returned: {result['source']}")
            print_state(state)
            continue

        # PATH 2: The model returned a finding.
        if decision.action == "final":

            # The harness enforces a requirement the model cannot waive:
            # both business sources must be inspected before accepting a finding.
            required_sources = set(TOOLS.keys())
            used_sources = set(state["sources_used"])

            if not required_sources.issubset(used_sources):
                print("Final answer rejected by harness.")
                print("Both Finance and Sales must be inspected first.")
                print()
                continue

            # The model interprets evidence and proposes the amount explained.
            state["explained_amount"] = int(
                decision.amount_usd or 0
            )

            # Python recalculates the residual from structured state.
            update_gap(state)

            state["findings"].append({
                "cause": decision.cause,
                "amount_usd": decision.amount_usd,
                "explanation": decision.explanation,
            })

            # Surface the actual business finding before the stop check.
            # This keeps the trace useful to a human: they can see not only
            # that the model found "something," but what it found and why.
            print("SUPPORTED FINDING")
            print(f"  Cause:       {decision.cause}")
            print(f"  Amount:      {money(decision.amount_usd)}")
            print(f"  Explanation: {decision.explanation}")
            print()

            # The stopping policy—not the model alone—decides whether this
            # business state is allowed to terminate the run.
            should_stop, status = stop_policy(
                state,
                decision,
            )

            print_state(state)
            print(f"STOP CHECK: {status}")
            print()

            if should_stop:
                return {
                    "state": state,
                    "decision": decision,
                    "status": status,
                    "evidence": evidence,
                }

    # Reaching the hard step limit is an operational terminal state.
    # It does NOT mean the reconciliation succeeded.
    return {
        "state": state,
        "decision": None,
        "status": "max_steps_reached",
        "evidence": evidence,
    }

### Run 1 — Inherited behavior

Run the cell below without changing the stopping policy.

### What to watch

You should see the agent:

1. inspect both available reporting sources;
2. establish the &#36;400,000 total discrepancy;
3. identify one supported refund-recognition timing cause;
4. show the **cause, amount, and evidence explanation** in a `SUPPORTED FINDING` block; and
5. ask the inherited stopping policy whether that finding is enough to terminate.

A detail that can look confusing at first:

After both sources are loaded but **before** the model returns its supported finding, the state can show:

- total discrepancy = &#36;400,000
- explained so far = &#36;0
- unexplained residual = &#36;400,000

That is correct. The harness knows the size of the gap, but no finding has yet been accepted into the business ledger.

Once the &#36;180,000 finding is accepted, the unexplained residual should become &#36;220,000.

Do not focus only on whether the finding itself is correct. Watch what the stopping policy does with that incomplete business state.

In [16]:
run_1 = run_agent(
    stop_policy=stop_when_model_has_supported_finding,
    max_steps=8,
)

STEP 1
Decision: load_finance_q3_revenue
Reason:   I need to establish the baseline revenue and refund treatment from the Finance department to begin the reconciliation process.

Tool executed: load_finance_q3_revenue
Source returned: Finance
STATE
  Finance reported:  $4,200,000
  Sales reported:    —
  Total discrepancy: —
  Explained so far:  $0
  Unexplained residual: —
  Sources used:      ['load_finance_q3_revenue']

STEP 2
Decision: load_sales_q3_revenue
Reason:   I have inspected the Finance data and now need to load the Sales Q3 revenue extract to compare the reported figures and identify the discrepancy.

Tool executed: load_sales_q3_revenue
Source returned: Sales
STATE
  Finance reported:  $4,200,000
  Sales reported:    $3,800,000
  Total discrepancy: $400,000
  Explained so far:  $0
  Unexplained residual: $400,000
  Sources used:      ['load_finance_q3_revenue', 'load_sales_q3_revenue']

STEP 3
Decision: final
Reason:   The discrepancy is caused by the Sales report includ

## 11. Inspect the Result as Structured State

A good agent engineer does not evaluate a run only by reading the final prose.

The prose may sound polished while hiding an incomplete or unsafe business state.

Structured state lets you inspect what the application actually believes happened:

- What were the authoritative source values?
- How large was the original discrepancy?
- How much did the accepted finding explain?
- How much is still unexplained?
- Were all currently available sources exhausted?
- Why did the harness stop?

This is also the kind of state you can test automatically. A sentence like “the investigation looks complete” is subjective. A residual of &#36;220,000 is machine-checkable.

The next cell prints the ledger explicitly and then surfaces the finding itself.

In [17]:
state_1 = run_1["state"]

print("RECONCILIATION STATE")
print("-" * 40)
print(f"Finance reported:      {money(state_1['finance_reported'])}")
print(f"Sales reported:        {money(state_1['sales_reported'])}")
print(f"Total discrepancy:     {money(state_1['total_gap'])}")
print(f"Explained:             {money(state_1['explained_amount'])}")
print(f"Unexplained residual:  {money(state_1['residual'])}")
print(f"Sources exhausted:     {sources_exhausted(state_1)}")
print(f"Stop status:           {run_1['status']}")

if state_1["findings"]:
    finding_1 = state_1["findings"][0]

    print()
    print("SUPPORTED FINDING DETAILS")
    print("-" * 40)
    print(f"Cause:       {finding_1['cause']}")
    print(f"Amount:      {money(finding_1['amount_usd'])}")
    print(f"Explanation: {finding_1['explanation']}")


RECONCILIATION STATE
----------------------------------------
Finance reported:      $4,200,000
Sales reported:        $3,800,000
Total discrepancy:     $400,000
Explained:             $180,000
Unexplained residual:  $220,000
Sources exhausted:     True
Stop status:           finding_returned

SUPPORTED FINDING DETAILS
----------------------------------------
Cause:       Sales report includes Q4-dated refunds (RF-1007 to RF-1013) in Q3 revenue, while Finance excludes them.
Amount:      $180,000
Explanation: The Sales report includes seven refunds (RF-1007, RF-1008, RF-1009, RF-1010, RF-1011, RF-1012, RF-1013) totaling $180,000 that occurred in Q4 (October 2026) but were attributed to Q3 revenue. Finance correctly excluded these from the Q3 report.


## 12. Engineering Checkpoint 3 — Diagnose and Fix Completion

Before moving on, separate **finding quality** from **completion quality**.

Answer:

1. How much of the &#36;400,000 discrepancy was explained?
2. How much remains unexplained?
3. Were all currently available sources used?
4. Was the refund-timing finding itself supported?
5. Why did the harness stop anyway?
6. Which component needs to change: the evidence, the model's interpretation, or the definition of completion?

The important engineering move is to change the smallest responsible component.

The evidence is fine. The supported finding is fine. The business stopping rule is the defect.

### YOUR TURN — Replace the Business Stopping Policy

Keep the same model, data, tools, instructions, and model decision contract. Change only the business stopping policy.

### COPY THIS

```python
def business_aware_stop(state, decision):
    if state["residual"] == 0:
        return True, "reconciled"

    if sources_exhausted(state):
        return True, "unresolved"

    return False, "continue"
```

### PASTE IT HERE

Paste the full function into the **next code cell**, directly below the `# PASTE HERE` comment, then run the cell.

This policy creates three business states:

- `reconciled` — the residual is zero; the business objective is complete;
- `unresolved` — the available sources are exhausted, but a residual remains;
- `continue` — more useful investigation work is still possible.

Notice that `unresolved` is a valid terminal state. Stopping honestly is better than manufacturing completion.

In [18]:
# YOUR TURN
# PASTE HERE
def business_aware_stop(state, decision):
    if state["residual"] == 0:
        return True, "reconciled"

    if sources_exhausted(state):
        return True, "unresolved"

    return False, "continue"



## 13. Run the Same Agent Again

Once you have defined `business_aware_stop`, run the next cell.

This is a controlled engineering experiment.

You are keeping all of these constant:

- Gemini model;
- system instructions;
- Finance and Sales data;
- tool functions;
- tool registry;
- dispatcher;
- state model.

You are changing **one thing only**: the stopping policy.

That isolation is useful because it lets you attribute the behavioral difference to the control you changed rather than to a new prompt, model, or dataset.

Watch the final terminal status closely.

The system is not expected to magically discover the missing &#36;220,000 in this lab. It does not have the additional evidence yet. The improvement is that it should describe its state truthfully.

In [19]:
run_2 = run_agent(
    stop_policy=business_aware_stop,
    max_steps=8,
)

STEP 1
Decision: load_finance_q3_revenue
Reason:   I need to establish the baseline revenue and refund treatment from the Finance department to begin the reconciliation process.

Tool executed: load_finance_q3_revenue
Source returned: Finance
STATE
  Finance reported:  $4,200,000
  Sales reported:    —
  Total discrepancy: —
  Explained so far:  $0
  Unexplained residual: —
  Sources used:      ['load_finance_q3_revenue']

STEP 2
Decision: load_sales_q3_revenue
Reason:   I have inspected the Finance data and now need to load the Sales Q3 revenue extract to compare the reported figures and identify the discrepancy.

Tool executed: load_sales_q3_revenue
Source returned: Sales
STATE
  Finance reported:  $4,200,000
  Sales reported:    $3,800,000
  Total discrepancy: $400,000
  Explained so far:  $0
  Unexplained residual: $400,000
  Sources used:      ['load_finance_q3_revenue', 'load_sales_q3_revenue']

STEP 3
Decision: final
Reason:   The discrepancy is caused by the Sales report includ

In [20]:
state_2 = run_2["state"]

print("FINAL RECONCILIATION STATUS")
print("-" * 40)
print(f"Finance reported:      {money(state_2['finance_reported'])}")
print(f"Sales reported:        {money(state_2['sales_reported'])}")
print(f"Total discrepancy:     {money(state_2['total_gap'])}")
print(f"Explained:             {money(state_2['explained_amount'])}")
print(f"Unexplained residual:  {money(state_2['residual'])}")
print(f"Sources exhausted:     {sources_exhausted(state_2)}")
print(f"Terminal status:       {run_2['status'].upper()}")

FINAL RECONCILIATION STATUS
----------------------------------------
Finance reported:      $4,200,000
Sales reported:        $3,800,000
Total discrepancy:     $400,000
Explained:             $180,000
Unexplained residual:  $220,000
Sources exhausted:     True
Terminal status:       UNRESOLVED


## 14. What Changed?

Compare Run 1 and Run 2.

The model did not change. The tools did not change. The evidence did not change. The supported &#36;180,000 finding did not change.

Only the harness's **definition of a valid terminal state** changed.

Run 1 effectively said:

> “I found one supported explanation, so I am done.”

Run 2 says:

> “I found a supported explanation, but the residual is still above zero. The currently available sources are exhausted, so I must stop as unresolved rather than claim reconciliation.”

This is a core agent-engineering lesson:

**A model can reason correctly and still participate in an unreliable system if the surrounding control logic defines success badly.**

An agent can stop correctly without succeeding completely.

## 15. Execution Safety Is a Different Problem

The stopping policy solves a **business correctness** problem:

> Has the task reached a valid business terminal state?

That is different from **runtime safety**:

> Is the agent behaving acceptably while trying to reach that state?

Those two layers should not be collapsed into one rule.

Examples of runtime safeguards include:

- maximum step counts;
- duplicate tool-call protection;
- token or cost budgets;
- timeouts;
- retry limits;
- circuit breakers;
- permission checks before a write.

Why separate them?

An agent could have an excellent definition of business success and still waste money calling the same tool twenty times. Or it could stay within eight steps while stopping on an incorrect business result.

One control protects the **quality of completion**. The other protects the **execution process**.

You already have a maximum-step cap. Next you will add a repeated-tool guard.

### YOUR TURN — Add a Repeated-Tool Guard

Now add an operational safeguard. This is separate from the business definition of done.

### COPY THIS

```python
def no_repeat_source_guard(tool_name, state):
    if tool_name in state["sources_used"]:
        return False, "duplicate_source_call"

    return True, "allowed"
```

### PASTE IT HERE

Paste the full function into the **next code cell**, directly below the `# PASTE HERE` comment, then run the cell.

This control does **not** decide whether the reconciliation is complete. It decides whether a proposed action is allowed to execute.


In [21]:
# YOUR TURN
# PASTE HERE
def no_repeat_source_guard(tool_name, state):
    if tool_name in state["sources_used"]:
        return False, "duplicate_source_call"

    return True, "allowed"


### Test the Safeguard Before Trusting It

Do not wait for a probabilistic model to misbehave before deciding whether your control works.

The next cell creates a deterministic test:

1. start with clean state;
2. mark the Finance source as already used;
3. ask the guard whether Finance may be called again;
4. assert that the answer is `False`.

This is ordinary software-engineering discipline applied to an agent system.

The model may be probabilistic. Your guardrail does not have to be.

In [22]:
# Create a deterministic test case for the repeated-tool guard.
test_state = new_state()
test_state["sources_used"].append(
    "load_finance_q3_revenue"
)

allowed, status = no_repeat_source_guard(
    "load_finance_q3_revenue",
    test_state,
)

print(f"Allowed: {allowed}")
print(f"Status:  {status}")

# A small assertion turns the safeguard into something we can verify automatically.
assert allowed is False
assert status == "duplicate_source_call"

print("Guard test passed.")

Allowed: False
Status:  duplicate_source_call
Guard test passed.


### Run 3 — Business Control + Operational Safeguard

Now run the agent with **both** layers enabled:

- `business_aware_stop` controls whether the business task may terminate; and
- `no_repeat_source_guard` controls whether an already-used source may execute again.

The goal is not to change the reconciliation answer.

The goal is to show that reliable agent behavior usually comes from **multiple small controls with different responsibilities**, rather than one giant prompt telling the model to “be careful.”

In [23]:
run_3 = run_agent(
    stop_policy=business_aware_stop,
    tool_guard=no_repeat_source_guard,
    max_steps=8,
)

print(f"Terminal status: {run_3['status'].upper()}")

STEP 1
Decision: load_finance_q3_revenue
Reason:   I need to establish the baseline revenue and refund treatment from the Finance department to begin the reconciliation process.

Tool executed: load_finance_q3_revenue
Source returned: Finance
STATE
  Finance reported:  $4,200,000
  Sales reported:    —
  Total discrepancy: —
  Explained so far:  $0
  Unexplained residual: —
  Sources used:      ['load_finance_q3_revenue']

STEP 2
Decision: load_sales_q3_revenue
Reason:   I have inspected the Finance data and now need to load the Sales Q3 revenue extract to compare the reported figures and identify the discrepancy.

Tool executed: load_sales_q3_revenue
Source returned: Sales
STATE
  Finance reported:  $4,200,000
  Sales reported:    $3,800,000
  Total discrepancy: $400,000
  Explained so far:  $0
  Unexplained residual: $400,000
  Sources used:      ['load_finance_q3_revenue', 'load_sales_q3_revenue']

STEP 3
Decision: final
Reason:   The discrepancy is caused by the Sales report includ

## 16. Engineering Checkpoint 4 — Separate the Controls

For each item below, notice who owns the decision in this lab.

| Decision or control | Owner |
|---|---|
| Choose which source to inspect next | Model |
| Retrieve Finance or Sales evidence | Tool / application |
| Calculate the &#36;400,000 gap | Deterministic application code |
| Interpret refund timing evidence | Model |
| Calculate the unexplained residual | Deterministic application code |
| Decide whether the residual is zero | Harness / business control logic |
| Block repeated calls to an unchanged source | Harness / operational safeguard |
| Cap the total number of execution steps | Harness / operational safeguard |
| Produce a concise business explanation | Model |

As an agent engineer, the design question is not simply:

> “What can the model do?”

It is:

> **“Which component should own each decision so the overall system is reliable, inspectable, and testable?”**

The model supplies judgment where judgment is useful. The harness owns controls that should be explicit, enforceable, and easy to verify.

## 17. Software Engineering Review

The notebook now uses several practices that matter in real agent systems.

### Clear responsibility boundaries

Tool functions retrieve data. The model proposes actions and interprets evidence. The dispatcher executes only registered actions. State carries business facts. The stopping policy governs completion.

When those responsibilities are mixed together, failures become much harder to diagnose.

### Structured model output

The model returns a defined schema instead of arbitrary prose when the application needs to make a control decision.

That makes outputs easier to validate and route.

### Explicit state

Important facts survive across steps in a machine-readable structure rather than depending on the model to remember them conversationally.

### Deterministic invariants

The total gap and unexplained residual are calculated in Python rather than repeatedly inferred from prose.

If a fact can be computed exactly, there is usually little value in making the model guess it.

### Bounded execution

`max_steps` prevents an unbounded loop even if another control fails.

This is a safety net, not a definition of success.

### Testable safeguards

The repeated-tool guard can be unit-tested directly.

Controls that can be tested independently are easier to trust than instructions that merely ask a model not to misbehave.

### Readable traces

The notebook prints decisions, actions, findings, and state transitions so a run can be inspected afterwards.

The more autonomy a system has, the more valuable that reconstruction becomes.

## 18. Finished Early? Knowledge Expanders

Complete the required lab first. Then choose one experiment.

### A. Lower the maximum step count

```python
run_agent(
    stop_policy=business_aware_stop,
    tool_guard=no_repeat_source_guard,
    max_steps=3,
)
```

What happens if an operational limit is reached before the task reaches a business terminal state?

### B. Remove the repeated-tool guard

Run the agent without `tool_guard`. Does the model repeat a source? If not, why is the guard still useful?

### C. Make the model own the residual

Temporarily remove the deterministic `update_gap()` calculation and ask the model to state the remaining residual. Compare the architecture, not only the answer.

### D. Add a new terminal status

What status would you return if a required tool failed instead of returning evidence? Examples: `tool_failure`, `needs_human_review`, or `blocked`.

## 19. Bridge to Lab 3

The Lab 2 agent now behaves more reliably, but it still cannot explain the full &#36;400,000 discrepancy.

It has:

- a &#36;400,000 total gap;
- &#36;180,000 explained by refund timing;
- a &#36;220,000 unexplained residual; and
- no additional sources available.

That is an important diagnosis.

The next engineering problem is **not**:

> “Make the loop longer.”

More iterations over the same evidence do not create new evidence.

The next question is:

> **What information does this investigation need that the current agent cannot reach?**

In Lab 3, you will expand the evidence surface and introduce specialist workers suited to different types of information.

## Lab 2 Completion Check

You are ready to move on when you can explain, in your own words:

- why the model is only one component of an agent system;
- what the harness controls around the model;
- how a tool registry defines available capabilities;
- why explicit state matters during multi-step execution;
- what the **unexplained residual** measures;
- why some business facts should remain deterministic;
- how a stopping condition changes agent behavior;
- why `unresolved` can be a correct terminal state;
- how business completion logic differs from runtime safeguards;
- why controls such as maximum-step limits and repeated-tool protection belong in the harness; and
- why changing the harness can change system reliability without changing the model.